### Here we will strat wit teh silver layer architecture : Red bronze - display brionze- rename columns - standarddize values- handling null - removing duplicates - data qulaity chceks - write silver delta

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    when,
    current_timestamp
)

In [0]:
bronze_df = (
    spark.read.format("Delta")\
        .load("/Volumes/workspace/default/my_volume/bronze")
)

In [0]:
display(bronze_df)

In [0]:
bronze_df.printSchema()

In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed("Item_Identifier", "item_id")
    .withColumnRenamed("Item_Weight", "item_weight")
    .withColumnRenamed("Item_Fat_Content", "item_fat_content")
    .withColumnRenamed("Item_Visibility", "item_visibility")
    .withColumnRenamed("Item_Type", "item_type")
    .withColumnRenamed("Item_MRP", "item_mrp")
    .withColumnRenamed("Outlet_Identifier", "outlet_id")
    .withColumnRenamed("Outlet_Establishment_Year", "outlet_establishment_year")
    .withColumnRenamed("Outlet_Size", "outlet_size")
    .withColumnRenamed("Outlet_Location_Type", "outlet_location_type")
    .withColumnRenamed("Outlet_Type", "outlet_type")
    .withColumnRenamed("Item_Outlet_Sales", "item_outlet_sales")
)

In [0]:
display(silver_df)

In [0]:
silver_df.printSchema()

### standardization

In [0]:
silver_df = silver_df.withColumn("item_fat_content", when(upper(trim(col("item_fat_content"))).isin("LF", "LOW FAT", "LOWFAT"), "Low fat")
                    .when(upper(trim(col("item_fat_content"))).isin("REG","REGULAR"), "Regular")
                    .otherwise("unknown"))

In [0]:
silver_df.select("item_fat_content").distinct().display()

### handle nulls

In [0]:
silver_df.filter(
    col("Item_weight").isNull()
).count()

In [0]:
silver_df.filter(
    col("Outlet_size").isNull()
).count()

## now here we got the count of null values and we will now start handling the nulls.

# this is for the outlet_size

In [0]:
silver_df = silver_df.fillna({
    "outlet_size" : "unknown"
})

In [0]:
median_weight = silver_df.approxQuantile(
    "item_weight",
    [0.5],
    0.01
)[0]

In [0]:
silver_df= silver_df.fillna({
    "item_weight" : median_weight
})

In [0]:
silver_df.filter(
    col("item_weight").isNull()
).count()

# REMOVE DUPLICATE RECORDS

In [0]:
before_count = silver_df.count()

In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
after_count=  silver_df.count()

In [0]:
print("Before duplicates removal", before_count)
print("After duplicates removal", after_count)

In [0]:
silver_df.display()

# another transformation is checking in the complete file whether i have any null values or not and if i do have then give me sum of null values of ecah column. 

In [0]:
from pyspark.sql.functions import col, sum

In [0]:
null_count = silver_df.select([
                sum(col(c).isNull().cast("int")).alias(c)
                for c in silver_df.columns

])

display(null_count)

In [0]:
required_columns = [
    "Item_Identifier",
    "Item_MRP",
    "Outlet_Type",
    "Item_Outlet_Sales"

]

for column_name in required_columns:
    null_count = silver_df.filter(
        col(column_name).isNull()
    ).count()

    print(f"{column_name}: {null_count} null values")


In [0]:
silver_df.filter(
    col("Item_MRP") <0
).count()

In [0]:
silver_df.filter(
    col("Item_Outlet_Sales") < 0
).count()

In [0]:
silver_df.filter(
    col("Item_Visibility") < 0
).count()

In [0]:
print("Silver record count" , silver_df.count())
print("Silver columns" , len(silver_df.columns))

In [0]:
silver_df.printSchema()

In [0]:
null_count = silver_df.filter(
    col("item_visibility").isNull()
).count()

In [0]:
print(f"Null count for item_visibility: {null_count}")

# all checks are done so we will be saving the silver as delta and then read and then verify and then we will be using the databricks to push into the github. 

In [0]:
silver_path = ("/Volumes/workspace/default/my_volume/silver")

In [0]:
silver_df.write\
    .format("delta")\
        .mode("overwrite") \
            .save(silver_path)


In [0]:
silver_check = spark.read\
    .format("delta")\
        .load("/Volumes/workspace/default/my_volume/silver")

In [0]:
display(silver_check)

In [0]:
silver_df.printSchema()